In [1]:
!kaggle datasets download -d "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:02<00:00, 11.2MB/s]



In [2]:
import zipfile
zip_ref = zipfile.ZipFile('/content/imdb-dataset-of-50k-movie-reviews.zip' , 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [3]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 14.2 MB/s eta 0:00:00


In [6]:
import contractions

text = "I don't like this movie. It's boring. I've seen better."

print(contractions.fix(text))

I do not like this movie. It is boring. I have seen better.


In [7]:
import re
import contractions

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Expand contractions
    text = contractions.fix(text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces
    text = ' '.join(text.split())

    return text

In [8]:
df["clean_review"] = df['review'].apply(preprocess_text)

In [ ]:
df['clean_review'].head()

,clean_review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production the filming tech...
2,i thought this was a wonderful way to spend ti...
3,basically there is a family where a little boy...
4,petter matteis love in the time of money is a ...


In [9]:
df['sentiment'] = df['sentiment'].map({'negative': 0, 'positive': 1})
# do not use label encoder??

In [10]:
df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,1,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,1,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,1,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,0,basically there is a family where a little boy...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter matteis love in the time of money is a ...


In [34]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(df['clean_review'])

sequences = tokenizer.texts_to_sequences(df['clean_review'])

print(sequences[0][:20])


[27, 4, 1, 81, 1879, 43, 1023, 11, 102, 145, 39, 3226, 379, 20, 54, 26, 3075, 29, 21, 200]


In [35]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(sequences, maxlen=200)

print(X.shape)

(50000, 200)


In [37]:
from sklearn.model_selection import train_test_split
import numpy as np
y = np.asarray(df['sentiment']).astype('int32')
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [38]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train,test_size=0.125, random_state=42,stratify=y_train)

In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense

model = Sequential([Embedding(input_dim=10000, output_dim=128),Bidirectional(LSTM(64)),Dense(1, activation='sigmoid')])

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
from tensorflow.keras.callbacks import (ModelCheckpoint,EarlyStopping,ReduceLROnPlateau)
model_name = "imdb_sentiment_model.keras"

checkpoint = ModelCheckpoint(
    model_name,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1)

earlystopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    verbose=1,
    restore_best_weights=True)

learning_rate_reduction = ReduceLROnPlateau(
    monitor="val_loss",
    patience=3,
    verbose=1,
    factor=0.2,
    min_lr=1e-7)


In [42]:
history = model.fit(X_train,y_train,validation_data=(X_val, y_val),epochs=20,batch_size=64,
            callbacks=[checkpoint,earlystopping,learning_rate_reduction])

Epoch 1/20
546/547 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7395 - loss: 0.4985
Epoch 1: val_loss improved from None to 0.33385, saving model to imdb_sentiment_model.keras

Epoch 1: finished saving model to imdb_sentiment_model.keras
547/547 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.8151 - loss: 0.4031 - val_accuracy: 0.8626 - val_loss: 0.3338 - learning_rate: 0.0010
Epoch 2/20
545/547 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8902 - loss: 0.2806
Epoch 2: val_loss improved from 0.33385 to 0.31196, saving model to imdb_sentiment_model.keras

Epoch 2: finished saving model to imdb_sentiment_model.keras
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.8930 - loss: 0.2711 - val_accuracy: 0.8792 - val_loss: 0.3120 - learning_rate: 0.0010
Epoch 3/20
546/547 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9295 - loss: 0.1922
Epoch 3: val_loss did not improve from 0.31196
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9268 - loss: 0.1961 - val_accuracy: 0.

In [43]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8826 - loss: 0.2937
Test Accuracy: 0.8826000094413757


In [46]:
loss, accuracy = model.evaluate(X_train, y_train)

print("Test Accuracy:", accuracy)

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.9377 - loss: 0.1699
Test Accuracy: 0.9376571178436279


In [49]:
def predict_sentiment(review):

    # Convert review to sequence
    sequence = tokenizer.texts_to_sequences([review])

    # Pad sequence
    padded_sequence = pad_sequences(sequence,maxlen=200)

    prediction = model.predict( padded_sequence,verbose=0)[0][0]

    # Convert probability to sentiment
    if prediction >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("Review:")
    print(review)

    print("\nPredicted Sentiment:", sentiment)
    print("Positive Probability:", round(float(prediction), 4))

In [50]:
review1 = """
This movie was absolutely fantastic. The acting was amazing
and the story kept me interested from beginning to end.
"""
predict_sentiment(review1)

Review:

This movie was absolutely fantastic. The acting was amazing
and the story kept me interested from beginning to end.


Predicted Sentiment: Positive
Positive Probability: 0.9067


In [51]:
review2 = """
I really hated this movie. The story was boring,
the acting was terrible, and I couldn't wait for it to end.
"""
predict_sentiment(review2)

Review:

I really hated this movie. The story was boring,
the acting was terrible, and I couldn't wait for it to end.


Predicted Sentiment: Negative
Positive Probability: 0.0514


In [53]:
review2 = "The movie was okay,nothing special"
predict_sentiment(review2)

Review:
The movie was okay,nothing special

Predicted Sentiment: Negative
Positive Probability: 0.159
